

# Approche 0.1: Détection de visage et organisation spatiale sur captures de vidéos



In [2]:
import numpy as np
import os
import gdown
from tqdm.notebook import tqdm
import cv2
#Si vous exécutez le code sur colab c'est une alternative pour imshow de cv2 qui n'y fonctionne pas
from google.colab.patches import cv2_imshow

In [3]:
HOME = os.getcwd()
print(HOME)

/content


In [4]:
VIDEO_DIR_PATH = f"{HOME}/videos"

In [5]:
!mkdir videos

In [6]:
# Récupérqtion de la vidéo de google drive
file_id = '1gH8WJqGdUvMLiAnfiLIEFR9c4rOQTBkn'
output_path = f'{VIDEO_DIR_PATH}/corrupted_video.mp4'
gdown.download(f'https://drive.google.com/uc?id={file_id}', output_path, quiet=False)

Downloading...
From: https://drive.google.com/uc?id=1gH8WJqGdUvMLiAnfiLIEFR9c4rOQTBkn
To: /content/videos/corrupted_video.mp4
100%|██████████| 4.84M/4.84M [00:00<00:00, 36.2MB/s]


'/content/videos/corrupted_video.mp4'

In [7]:
# Chargement du modèle préentrainé Haarcascades face detector
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')



In [8]:
# Lecture de la vidéo
video_path = f'{VIDEO_DIR_PATH}/corrupted_video.mp4'
cap = cv2.VideoCapture(video_path)

# Création d'un objet videowriter qui permettra plutard d'enregistrer le resultat dans une video
# Initialisation avec les caractéristiques de la vidéo corrompue
output_path = f'{VIDEO_DIR_PATH}/ordered_video.mp4'
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
video_writer = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))


In [9]:
# Liste vide qui stockera plutard les position de visage dans chaque cadre et l'indice qui identifiera ce cadre
frame_face_positions = []

# Parcours des cadres
frame_index = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Détection faciale sure chque cadre
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.3, minNeighbors=9)

    if len(faces) > 0:
        # Stockage de l'indice du frame et et de coordonnése spatiaux dans la liste
        frame_face_positions.append((frame_index, faces[0][0]))

    frame_index += 1


In [10]:
# Ordonnancement des cadres selon la position spatiale du visage
frame_face_positions.sort(key=lambda x: x[1], reverse=True)

In [11]:
# Réiniitiqlisqtion du lecteur vidéo pour réarranger les cadres
cap = cv2.VideoCapture(video_path)

# Parcours des cadres
for frame_index, _ in frame_face_positions:
  # définition du curseur par ordre défini plutot
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_index)
    #lecture du cadre pointé par le curseur
    ret, frame = cap.read()
    #vérification de la lecture
    if not ret:
        break

    # Ecriture des cadres ordonnés dans une nouvelle vidéo
    video_writer.write(frame)

In [13]:
# Libération des objets et curseurs
cap.release()
video_writer.release()
cv2.destroyAllWindows()